## Cell 1

In [ ]:
!pip install -q python-docx gradio

## Cell 2

In [ ]:
import re
import unicodedata
import gradio as gr

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00a0", " ")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text).strip().casefold()

print("Application utilities ready!")

## Cell 3

In [ ]:
!pip install -q google-api-python-client

from google.colab import auth
import google.auth
from googleapiclient.discovery import build

# Sign in to your Google account
auth.authenticate_user()

# Obtain authenticated credentials
credentials, project = google.auth.default()

# Connect to Google Drive
drive_service = build(
    "drive",
    "v3",
    credentials=credentials
)

# Test the connection
account = drive_service.about().get(
    fields="user"
).execute()

print("Google authentication successful!")
print("Connected account:", account["user"]["emailAddress"])

## Cell 4

In [ ]:
import re
import os

def download_google_doc(doc_url):

    # Extract document ID from Google Docs URL
    match = re.search(r'/document/d/([a-zA-Z0-9_-]+)', doc_url)

    if not match:
        raise ValueError("Invalid Google Docs URL.")

    doc_id = match.group(1)

    # Retrieve document information
    file_info = drive_service.files().get(
        fileId=doc_id,
        fields="id,name,mimeType"
    ).execute()

    if file_info["mimeType"] != "application/vnd.google-apps.document":
        raise ValueError("This is not a Google Docs document.")

    # Export document as Word
    content = drive_service.files().export_media(
        fileId=doc_id,
        mimeType="application/vnd.openxmlformats-officedocument.wordprocessingml.document"
    ).execute()

    # Save converted document
    output_path = "/content/source_document.docx"

    with open(output_path, "wb") as f:
        f.write(content)

    print("Document retrieved successfully!")
    print("Original name:", file_info["name"])
    print("Converted to: DOCX")

    return output_path


print("Google Docs retrieval function created!")

## Cell 5

In [ ]:
from docx import Document
from docx.oxml.ns import qn
from docx.text.paragraph import Paragraph

def extract_section(file_path, first_line, last_line):
    doc = Document(file_path)
    body = doc.element.body

    # Get the main document blocks, excluding section settings.
    blocks = [
        element for element in body
        if element.tag != qn("w:sectPr")
    ]

    first = normalize_text(first_line)
    last = normalize_text(last_line)

    def find_block(target):
        matches = []

        for block_index, block in enumerate(blocks):
            for paragraph_element in block.iter(qn("w:p")):
                paragraph = Paragraph(paragraph_element, doc)

                if normalize_text(paragraph.text) == target:
                    matches.append(block_index)

        matches = sorted(set(matches))

        if not matches:
            raise ValueError(
                "A boundary paragraph could not be found."
            )

        if len(matches) > 1:
            raise ValueError(
                "A boundary paragraph appears more than once."
            )

        return matches[0]

    start = find_block(first)
    end = find_block(last)

    if start > end:
        raise ValueError(
            "The starting paragraph occurs after the ending paragraph."
        )

    # Keep the selected blocks in this in-memory document only.
    selected_blocks = set(blocks[start:end + 1])

    for element in list(body):
        if (
            element.tag != qn("w:sectPr")
            and element not in selected_blocks
        ):
            body.remove(element)

    output_path = "/content/extracted_section.docx"
    doc.save(output_path)

    return output_path, end - start + 1

print("New extraction engine ready!")

## Cell 6

In [ ]:
from docx import Document
from docx.oxml.ns import qn
from docx.text.paragraph import Paragraph

def create_remaining_copy(file_path, first_line, last_line):
    doc = Document(file_path)
    body = doc.element.body

    blocks = [
        element for element in body
        if element.tag != qn("w:sectPr")
    ]

    def find_block(target):
        matches = []

        for i, block in enumerate(blocks):
            for p in block.iter(qn("w:p")):
                if normalize_text(Paragraph(p, doc).text) == target:
                    matches.append(i)

        matches = sorted(set(matches))

        if len(matches) != 1:
            raise ValueError(
                "A boundary was missing or appeared more than once."
            )

        return matches[0]

    start = find_block(normalize_text(first_line))
    end = find_block(normalize_text(last_line))

    if start > end:
        raise ValueError("The boundaries are in the wrong order.")

    for block in blocks[start:end + 1]:
        body.remove(block)

    output_path = "/content/document_without_extracted_section.docx"
    doc.save(output_path)

    return output_path

remaining_path = create_remaining_copy(
    source_path,
    first_boundary,
    last_boundary
)

print("Remaining-document copy created!")
print(remaining_path)

## Cell 7

In [ ]:
def process_document(doc_url, first_line, last_line):
    if not doc_url.strip():
        raise gr.Error("Please enter a Google Docs URL.")

    if not first_line.strip() or not last_line.strip():
        raise gr.Error("Please enter both boundary paragraphs.")

    try:
        source_path = download_google_doc(doc_url)

        extracted_path, count = extract_section(
            source_path,
            first_line,
            last_line
        )

        remaining_path = create_remaining_copy(
            source_path,
            first_line,
            last_line
        )

        return (
            extracted_path,
            remaining_path,
            f"Success! Extracted {count} document blocks. "
            "Both documents have been created."
        )

    except Exception as e:
        raise gr.Error(str(e))


with gr.Blocks(title="Document Section Extractor") as app:

    gr.Markdown("# Document Section Extractor")
    gr.Markdown(
        "Extract a section from Google Docs and create "
        "a separate copy with that section removed."
    )

    doc_url = gr.Textbox(label="Google Docs URL")

    first_line = gr.Textbox(
        label="First paragraph of the section",
        lines=3
    )

    last_line = gr.Textbox(
        label="Last paragraph of the section",
        lines=3
    )

    extract_button = gr.Button(
        "Extract Section",
        variant="primary"
    )

    status = gr.Textbox(
        label="Status",
        interactive=False
    )

    extracted_file = gr.File(
        label="1. Extracted Section"
    )

    remaining_file = gr.File(
        label="2. Document Without Extracted Section"
    )

    extract_button.click(
        fn=process_document,
        inputs=[doc_url, first_line, last_line],
        outputs=[extracted_file, remaining_file, status]
    )

app.launch(share=True)